# 🚀 MultiTask GRU Systematic Sweep Analysis

Интерактивный анализ результатов полного цикла экспериментов с **MultiTask GRU**:
1. **Этап A**: Длина истории (L44, L60, L90, L120, L180)
2. **Этап B**: Охват якорей (`recent_14` vs `all_existing_22`)
3. **Этап C**: Полный календарь с весенним якорем `2025-02-13` и каналом `is_observed`
4. **Этап D**: Гиперпараметры (Hidden size, layers, dropout, learning rate, loss weights)
5. **Этап E**: Устойчивость по 3 seeds (42, 43, 44) и валидация на 4-х временных бэктестах

In [ ]:
import polars as pl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

REGISTRY_PATH = Path('artifacts/gru_sweep/experiment_registry.csv')
if REGISTRY_PATH.exists():
    df_reg = pl.read_csv(REGISTRY_PATH).to_pandas()
    print(f'[+] Загружено {len(df_reg)} экспериментов из реестра!')
    display(df_reg.head())
else:
    print('[-] Реестр пока пуст или формируется.')

## 1. Сравнение длины истории (Этап A: L44 to L180)

In [ ]:
if REGISTRY_PATH.exists():
    df_len = df_reg[df_reg['anchor_set'] == 'recent_14'].sort_values('sequence_length')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    ax1.plot(df_len['sequence_length'], df_len['rmsle_blend_cb'], 'o-', color='#2b5c8f', lw=2.5, ms=8, label='Blend RMSLE (50% CB + 50% GRU)')
    ax1.plot(df_len['sequence_length'], df_len['rmsle_factorized'], 's--', color='#e26d5c', lw=2, ms=7, label='Solo GRU Factorized RMSLE')
    ax1.set_title('Качество (RMSLE) vs Длина истории (дни)', fontweight='bold')
    ax1.set_xlabel('Длина последовательности (дней)')
    ax1.set_ylabel('RMSLE (меньше = лучше)')
    ax1.legend()
    
    ax2.plot(df_len['sequence_length'], df_len['reactivation_auc'], 'o-', color='#38b000', lw=2.5, label='Reactivation AUC')
    ax2.plot(df_len['sequence_length'], df_len['churn_auc'], '^-', color='#7209b7', lw=2.5, label='Churn AUC')
    ax2.set_title('AUC классификации переходов vs Длина истории', fontweight='bold')
    ax2.set_xlabel('Длина последовательности (дней)')
    ax2.set_ylabel('ROC-AUC (больше = лучше)')
    ax2.legend()
    
    plt.tight_layout()
    plt.show()

## 2. Охват якорей (`recent_14` vs `all_existing_22` vs `full_calendar`)

In [ ]:
if REGISTRY_PATH.exists():
    df_anchors = df_reg[df_reg['run_id'].str.contains('recent14|all22|full_calendar')]
    plt.figure(figsize=(10, 5))
    sns.barplot(data=df_anchors, x='anchor_set', y='rmsle_blend_cb', hue='sequence_length', palette='viridis')
    plt.title('Сравнение стратегий покрытия якорей по Blend RMSLE', fontweight='bold')
    plt.ylabel('Blend RMSLE')
    plt.ylim(df_anchors['rmsle_blend_cb'].min() - 0.005, df_anchors['rmsle_blend_cb'].max() + 0.005)
    plt.show()

## 3. Результаты 4-х временных бэктестов

In [ ]:
BT_PATH = Path('artifacts/gru_sweep/four_backtests_summary.csv')
if BT_PATH.exists():
    df_bt = pl.read_csv(BT_PATH).to_pandas()
    display(df_bt)
    
    plt.figure(figsize=(12, 5))
    plt.plot(df_bt['anchor_date'], df_bt['rmsle_factorized'], 'o-', color='#e63946', lw=2.5, ms=8, label='RMSLE')
    plt.title('Устойчивость финальной MultiTask GRU на 4-х временных бэктестах', fontweight='bold')
    plt.xlabel('Якорная дата бэктеста')
    plt.ylabel('RMSLE')
    plt.legend()
    plt.show()
else:
    print('[-] Бэктесты еще выполняются.')